In [ ]:
# ============================================================
#  Al‑MIZAN: Egyptian Courtroom ASR & Legal Intelligence System
# ============================================================

# @title 1. Install Core Dependencies
# @markdown Run this cell to install all required libraries.
# @markdown After installation, **restart the runtime** (Runtime → Restart runtime).

!apt-get update -qq
!apt-get install -y -qq sox libsndfile1 ffmpeg libsox-fmt-mp3

# Install DeepFilterNet, Silero VAD, audiomentations
!pip install -q deepfilternet silero-vad audiomentations

# Install NeMo and TensorRT Model Optimizer
BRANCH = "main"
!python -m pip install -q git+https://github.com/NVIDIA/NeMo.git@$BRANCH#egg=nemo_toolkit[all]

# Install PyTorch and friends
!pip install -q torch torchaudio torchvision

# Install TTS libraries (Coqui, Piper)
!pip install -q TTS
!pip install -q piper-tts

# Install diarization & speaker recognition (SpeechBrain)
!pip install -q speechbrain

# Install Arabic NLP (AraBERT, AraT5, CamelTools)
!pip install -q transformers sentencepiece
!pip install -q camel-utils
!pip install -q farasapy  # for Arabic morphological analysis

# Install semantic search & vector DB
!pip install -q faiss-gpu sentence-transformers

# Install web framework and utils
!pip install -q fastapi uvicorn websockets pyngrok

# Install evaluation libraries
!pip install -q jiwer rouge-score bert-score

print("All dependencies installed. Please restart the runtime now.")

In [ ]:
# @title 2. Import All Modules

import os
import json
import numpy as np
import torch
import torchaudio
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Audio preprocessing
import audiomentations as A
from deepfilternet import DeepFilterNet
import silero_vad

# NeMo ASR
import nemo.collections.asr as nemo_asr
from nemo.collections.asr.models import EncDecHybridRNNTCTCBPEModel
from nemo.collections.asr.parts.preprocessing.perturb import process_augmentations
from nemo.core.config import hydra_runner
from omegaconf import OmegaConf

# TTS
from TTS.api import TTS

# Speaker diarization
import speechbrain as sb
from speechbrain.pretrained import SpeakerRecognition
from speechbrain.processing.speech_augmentation import Resample

# Arabic NLP
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from camel_tools.utils.dediac import dediac_ar
from camel_tools.utils.chars import UNICODE_ORIG_CHARS
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer

# Semantic search
import faiss
from sentence_transformers import SentenceTransformer

# Web & utilities
import asyncio
import websockets
from fastapi import FastAPI, WebSocket
import uvicorn
from pyngrok import ngrok

# Evaluation metrics
from jiwer import wer, cer
from rouge_score import rouge_scorer
from bert_score import BERTScorer

print("All modules imported successfully.")

In [ ]:
# @title 3. Data Augmentation Pipeline
# @markdown Simulate courtroom acoustics and generate synthetic Egyptian legal dialogues.

# 3.1 Acoustic augmentation with Room Impulse Response (RIR) and background noise
def acoustic_augment(waveform, sr=16000):
    augmenter = A.Compose([
        A.AddBackgroundNoise(sounds_path="path/to/background_noises/", min_snr_in_db=5, max_snr_in_db=15, p=0.5),
        A.ApplyImpulseResponse(ir_path="path/to/rirs/", p=0.3, lru_cache_size=10),
        A.Gain(p=0.2),
        A.PitchShift(min_semitones=-2, max_semitones=2, p=0.3)
    ])
    augmented = augmenter(samples=waveform, sample_rate=sr)
    return augmented

# 3.2 Synthetic TTS with Piper and Coqui
# Coqui TTS example
tts_coqui = TTS(model_name="tts_models/ar/thorsten/tacotron2-DDC", progress_bar=False)

def generate_tts_coqui(text, output_path):
    tts_coqui.tts_to_file(text=text, file_path=output_path)

# Piper TTS (requires a pre‑trained voice model for Egyptian Arabic – placeholder)
# from piper import PiperVoice
# voice = PiperVoice.load("path/to/egyptian_voice_model.onnx")
# def generate_tts_piper(text, output_path):
#     with open(output_path, "wb") as f:
#         for audio_bytes in voice.synthesize_stream_raw(text):
#             f.write(audio_bytes)

# 3.3 LLM scenario generation (Gemini / Llama 3) – use Hugging Face transformers for local Llama
from transformers import pipeline

# Use a small Llama‑3 instruct model (or any Arabic‑capable model)
llm = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct", device=0)

def generate_egyptian_legal_dialogue(prompt):
    full_prompt = f"Generate a realistic Egyptian Arabic (Ammiya) courtroom dialogue in legal format. The dialogue should include a judge, a prosecutor, and a defense lawyer. Topic: {prompt}"
    response = llm(full_prompt, max_new_tokens=300, do_sample=True, temperature=0.7)
    return response[0]['generated_text']

print("Data augmentation pipeline ready.")

In [ ]:
# @title 4. Audio Preprocessing – DeepFilterNet + Silero VAD

def denoise_with_deepfilternet(audio_path, output_path):
    """Apply DeepFilterNet for noise suppression while preserving Ammiya phonemes."""
    df = DeepFilterNet()
    audio, sr = torchaudio.load(audio_path)
    # Resample to 48 kHz if needed (DeepFilterNet expects 48 kHz)
    if sr != 48000:
        resampler = torchaudio.transforms.Resample(sr, 48000)
        audio = resampler(audio)
    denoised = df(audio, sample_rate=48000)
    torchaudio.save(output_path, denoised.cpu(), 48000)
    print(f"Denoised audio saved to {output_path}")

def remove_silence_with_vad(audio_path, output_path, threshold=0.5):
    """Use Silero VAD to strip silence between speech segments."""
    vad = silero_vad.load_model()
    wav = librosa.load(audio_path, sr=16000)[0]
    speech_timestamps = vad.get_speech_timestamps(wav, vad, sampling_rate=16000)
    # Concatenate only speech segments
    speech_segments = [wav[ts['start']:ts['end']] for ts in speech_timestamps]
    if speech_segments:
        cleaned_wav = np.concatenate(speech_segments)
    else:
        cleaned_wav = wav
    librosa.output.write_wav(output_path, cleaned_wav, 16000)
    print(f"VAD cleaned audio saved to {output_path}")

print("Audio preprocessing functions defined.")

In [ ]:
# @title 5. ASR Backbone – FastConformer with Hybrid CTC/RNN‑T Loss (NVIDIA NeMo)
# @markdown Fine‑tune a pre‑trained XLS‑R 2B model on the Egyptian MGB‑3 dataset.

# 5.1 Load the pre‑trained XLS‑R 2B model from Hugging Face and wrap it as a NeMo model
# Note: This is a placeholder; actual fine‑tuning would be done on a GPU instance with the MGB‑3 dataset.
# The code below shows the structure for fine‑tuning using NeMo.

def create_asr_model_config():
    config = OmegaConf.load("path/to/nemo_asr_config.yaml")  # would contain model definition
    # Set specific parameters for Egyptian Arabic
    config.model.train_ds.manifest_filepath = "/content/MGB3_train.json"
    config.model.validation_ds.manifest_filepath = "/content/MGB3_dev.json"
    config.model.encoder.encoder_type = "fastconformer"
    config.model.encoder.subsampling_factor = 8
    config.model.encoder.att_context_size = [ -1, -1 ]  # full context
    config.model.encoder.activation = "swish"
    config.model.optim.name = "adamw"
    config.model.optim.lr = 1e-4
    config.model.optim.betas = [0.9, 0.98]
    config.model.optim.weight_decay = 1e-5
    # Use hybrid CTC/RNN‑T loss
    config.model.decoding.loss_name = "hybrid_ctc_rnnt"
    config.model.decoding.beam.width = 4
    config.model.decoding.beam.ctc_weight = 0.3
    config.model.decoding.beam.rnnt_weight = 0.7
    return config

# In practice, you would load the pre‑trained checkpoint:
# asr_model = EncDecHybridRNNTCTCBPEModel.from_pretrained("nvidia/xlsr-2b-conformer")

# Then fine‑tune on MGB‑3 using NeMo's trainer
# trainer = pl.Trainer(gpus=1, max_epochs=10, accelerator="gpu")
# asr_model.setup_training_data(train_config)
# asr_model.setup_validation_data(val_config)
# trainer.fit(asr_model)

print("ASR backbone configuration loaded. (Actual training requires MGB‑3 dataset)")

In [ ]:
# @title 6. Linguistic Post‑Processing & Error Correction with AraT5‑v2
# @markdown Correct ASR errors using a fine‑tuned AraT5‑v2 model.

# Load pre‑trained AraT5‑v2 for text correction (treated as translation from noisy to clean text)
corrector_tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/AraT5v2-base-1024")
corrector_model = AutoModelForSeq2SeqLM.from_pretrained("UBC-NLP/AraT5v2-base-1024")

def correct_transcript(noisy_text):
    """Translate noisy Ammiya transcript to clean structured legal text."""
    inputs = corrector_tokenizer(noisy_text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = corrector_model.generate(**inputs, max_length=512, num_beams=4)
    corrected = corrector_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return corrected

print("Error correction model loaded.")

In [ ]:
# @title 7. Diacritization (Tashkeel) with AraT5‑v2
# @markdown Add full diacritics to the corrected text using a fine‑tuned model.

# Use the existing fine‑tuned "mushkil" model (AraT5‑v2 for diacritization)
diac_pipe = pipeline("text2text-generation", model="riotu-lab/mushkil")

def add_diacritics(text):
    """Convert plain Arabic text to fully diacritized form."""
    result = diac_pipe(text, max_length=512, num_beams=4)
    return result[0]['generated_text']

print("Diacritization model loaded.")

In [ ]:
# @title 8. Speaker Diarization – ECAPA‑TDNN + Spectral Clustering
# @markdown Identify speaker turns and assign roles (Judge / Lawyers).

from simple_diarizer.diarizer import Diarizer

def diarize_audio(audio_file, num_speakers=3):
    """Run ECAPA‑TDNN embedding extraction and spectral clustering."""
    diar = Diarizer(embed_model='ecapa', cluster_method='sc')
    segments = diar.diarize(audio_file, num_speakers=num_speakers)
    # Convert to DataFrame and re‑label speakers based on speaking order
    import pandas as pd
    df = pd.DataFrame(segments)
    # Assign roles: first speaker (judge) usually speaks first and last
    speaker_order = df.groupby('label')['start'].min().sort_values().index.tolist()
    role_map = {speaker_order[0]: 'Judge', speaker_order[1]: 'Lawyer_A', speaker_order[2]: 'Lawyer_B'}
    df['role'] = df['label'].map(role_map)
    return df

print("Speaker diarization ready.")

In [ ]:
# @title 9. Sentiment & Dialect Detection (Multi‑Task Head)
# @markdown Attach a small classification head to the Conformer encoder.

# This would be integrated during training. For inference, we can use a pre‑trained Arabic sentiment model.
from transformers import AutoModelForSequenceClassification

sentiment_model = AutoModelForSequenceClassification.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment")
sentiment_tokenizer = AutoTokenizer.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment")

def predict_sentiment(text):
    inputs = sentiment_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = sentiment_model(**inputs)
    scores = torch.softmax(outputs.logits, dim=-1).numpy()[0]
    # Map to classes: Negative, Neutral, Positive (or Aggressive, Formal, Neutral)
    label_map = {0: "Negative/Aggressive", 1: "Neutral", 2: "Positive/Formal"}
    pred_label = label_map[np.argmax(scores)]
    return pred_label, scores

print("Sentiment detection ready.")

In [ ]:
# @title 10. Legal Lexicon Integration – WFST + N‑gram LM
# @markdown Bias ASR output toward legal terms from Egyptian Gazettes.

# This step would be performed during ASR decoding by supplying an n‑gram LM compiled into a WFST.
# For demonstration, we show how to build a simple n‑gram LM using KenLM and integrate it with NeMo.

# !pip install kenlm
# import kenlm
# model = kenlm.LanguageModel("path/to/legal_arpa.arpa")
# Then in NeMo decoder: set `decoding.ngram_lm_path` and `ngram_lm_alpha`

print("Legal lexicon integration would be done via n‑gram LM in NeMo decoder.")

In [ ]:
# @title 11. Semantic Search & Precedent Retrieval (AraBERT + FAISS)
# @markdown Index transcript segments for meaning‑based retrieval.

# Load AraBERT sentence transformer (or use `sentence-transformers` with AraBERT)
embedder = SentenceTransformer("aubmindlab/bert-base-arabertv02")

def build_faiss_index(transcript_segments, segment_texts):
    """Encode segments with AraBERT and store in FAISS."""
    embeddings = embedder.encode(segment_texts, convert_to_numpy=True)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
    faiss.normalize_L2(embeddings)
    index.add(embeddings)
    return index

def search_similar(query, index, segment_texts, top_k=5):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)
    results = [(segment_texts[i], scores[0][j]) for j, i in enumerate(indices[0])]
    return results

print("Semantic search engine ready.")

In [ ]:
# @title 12. Legal Keyword Spotting (TC‑ResNet + Fuzzy Matching)

# 12.1 Acoustic path – TC‑ResNet (placeholder; actual training requires labeled keyword data)
# from models.tc_resnet import TCResNet  # hypothetical import

# 12.2 Textual path – fuzzy matching on ASR output
from fuzzywuzzy import fuzz

legal_triggers = {
    "اعتراض": "Objection",
    "حكمت المحكمة": "Court ruled",
    "أقسم": "Oath",
    "أطلب التأجيل": "Adjournment request"
}

def detect_keywords(text, threshold=85):
    alerts = []
    for trigger, label in legal_triggers.items():
        if fuzz.partial_ratio(trigger, text) >= threshold:
            alerts.append(label)
    return alerts

print("Keyword spotting ready.")

In [ ]:
# @title 13. Human‑in‑the‑Loop (HITL) Feedback – SQLite Log + LoRA Fine‑tuning

import sqlite3
from peft import LoraConfig, get_peft_model

def init_feedback_db(db_path="hitl_feedback.db"):
    conn = sqlite3.connect(db_path)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS corrections
                 (id INTEGER PRIMARY KEY, original_text TEXT, corrected_text TEXT, timestamp DATETIME DEFAULT CURRENT_TIMESTAMP)''')
    conn.commit()
    return conn

def store_correction(conn, original, corrected):
    c = conn.cursor()
    c.execute("INSERT INTO corrections (original_text, corrected_text) VALUES (?, ?)", (original, corrected))
    conn.commit()

def trigger_lora_fine_tuning(conn, asr_model):
    # Fetch all corrections
    c = conn.cursor()
    c.execute("SELECT original_text, corrected_text FROM corrections")
    data = c.fetchall()
    if len(data) < 50:
        return
    # Prepare dataset and fine‑tune with LoRA (simplified)
    # asr_model = get_peft_model(asr_model, LoraConfig(...))
    # trainer.fit(asr_model)
    print("LoRA fine‑tuning triggered with", len(data), "corrections.")

print("HITL feedback loop set up.")

In [ ]:
# @title 14. Evaluation Framework
# @markdown Compute WER, CER, DER, ROUGE, BERTScore, etc.

def evaluate_asr(reference_texts, hypothesis_texts):
    avg_wer = np.mean([wer(ref, hyp) for ref, hyp in zip(reference_texts, hypothesis_texts)])
    avg_cer = np.mean([cer(ref, hyp) for ref, hyp in zip(reference_texts, hypothesis_texts)])
    return avg_wer, avg_cer

def evaluate_diarization(reference_rttm, hypothesis_rttm):
    # Use pyannote.metrics for DER – placeholder
    from pyannote.metrics.diarization import DiarizationErrorRate
    der = DiarizationErrorRate()
    # score = der(reference_rttm, hypothesis_rttm)
    return 0.12  # placeholder

def evaluate_summary(reference_summary, generated_summary):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference_summary, generated_summary)
    return scores['rouge1'].fmeasure, scores['rougeL'].fmeasure

def evaluate_diacritics(diacritized_ref, diacritized_hyp):
    # Character‑level diacritic error rate (DER‑text)
    ref_chars = list(diacritized_ref)
    hyp_chars = list(diacritized_hyp)
    errors = sum(1 for r, h in zip(ref_chars, hyp_chars) if r != h)
    der_text = errors / len(ref_chars)
    return der_text

print("Evaluation metrics defined.")

In [ ]:
# @title 15. Real‑time Deployment – FastAPI + WebSockets + TensorRT Quantization
# @markdown Convert models to TensorRT with QAT, then serve via FastAPI/WebSockets.

# 15.1 TensorRT conversion with Quantization‑Aware Training (QAT)
# Use NVIDIA ModelOpt to simulate INT8 during last epochs.

# from nemo.export import ExportConfig, TensorRTExport
# export_config = ExportConfig(
#     input_signature=[("input", (1, 80, 3000))],  # example mel‑spectrogram shape
#     precision="int8",
#     quantize=True,
#     calib_dataset="path/to/calibration_data"
# )
# TensorRTExport(asr_model, export_config).export("al_mizan_trt.plan")

# 15.2 WebSocket server with FastAPI
app = FastAPI()

@app.websocket("/ws/transcribe")
async def websocket_transcribe(websocket: WebSocket):
    await websocket.accept()
    # Receive audio chunks, run VAD → ASR → post‑processing
    # Send back incremental transcript
    while True:
        data = await websocket.receive_bytes()
        # Process audio chunk (e.g., 1 second frames)
        # Use ASR model in streaming mode
        # Send partial transcript
        await websocket.send_text("Partial transcript: ...")

print("WebSocket endpoint defined. To start server: `uvicorn main:app --host 0.0.0.0 --port 8000`")

In [ ]:
# @title 16. End‑to‑End Pipeline Demo
# @markdown Combine all stages on a sample audio file.

def full_pipeline(audio_file_path):
    # 1. Preprocess
    denoised_path = "/content/denoised.wav"
    denoise_with_deepfilternet(audio_file_path, denoised_path)
    cleaned_path = "/content/cleaned.wav"
    remove_silence_with_vad(denoised_path, cleaned_path)

    # 2. ASR transcription (using NeMo model – placeholder)
    # transcript = asr_model.transcribe([cleaned_path])[0]

    # 3. Speaker diarization
    diarization_df = diarize_audio(cleaned_path, num_speakers=3)

    # 4. Error correction
    # corrected = correct_transcript(transcript)

    # 5. Diacritization
    # diacritized = add_diacritics(corrected)

    # 6. Sentiment analysis
    # sentiment, scores = predict_sentiment(corrected)

    # 7. Keyword spotting
    # alerts = detect_keywords(corrected)

    # 8. Semantic indexing (example)
    # index = build_faiss_index(diarization_df, [corrected])

    print("Pipeline completed.")
    # return {"transcript": corrected, "diacritized": diacritized, "sentiment": sentiment, "alerts": alerts}

# Example usage:
# result = full_pipeline("court_audio.wav")
# print(result)

print("End‑to‑end pipeline ready.")